# Industrial solver audit

Compare industrial solvers separately for discounted, total and average reward, at tiny, small, medium and large catalogue sizes. Average reward reuses discounted models; total reward uses its own catalogue. Results from different criteria or sizes are not pooled.

Each run saves a CSV and its experiment manifest, then shows coverage, performance profiles and runtime heatmaps. Summary rows are ordered by models solved, share within twice the fastest solver, then median runtime ratio. These describe execution performance, not certified accuracy.

In [ ]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

from IPython.display import display

from mdpforge import Benchmark
from mdpforge.solvers import average, discounted, total

DISCOUNT = 0.99
PRECISION = 1e-3
REPEATS = 5
SEED = 0
TIMEOUTS = {"tiny": 10, "small": 30, "medium": 60, "large": 360}
VALIDATE = False

BACKENDS = [
    "marmote",
    "mdpsolver",
    "mdptoolbox",
    "madupite",
    "mdpax",
    "quantecon",
    "scipy",
    "pyomo",
]
SOLVER_CATALOGUES = {
    "discounted": discounted,
    "total": total,
    "average": average,
}
results = {}

## Backend coverage and validation

The table below reports catalogue support, independently of installed dependencies. Missing optional dependencies produce warnings during solver registration; the saved manifest lists the solvers actually registered. Unsupported backends are excluded from that criterion's ranking.

Set `VALIDATE = True` to request certification after timing for discounted and total reward. Total certification currently covers absorbing acyclic models; average certification is unavailable and always disabled. Solver stopping tolerances may have different meanings; execution success alone does not establish equal solution quality.

In [ ]:
backends_by_criterion = {}
for criterion, catalogue in SOLVER_CATALOGUES.items():
    entries = catalogue.list_solvers()
    supported = {entry["backend"] for entry in entries}
    backends_by_criterion[criterion] = [
        backend for backend in BACKENDS if backend in supported
    ]
    print(f"\n{criterion}")
    for backend in BACKENDS:
        names = [entry["name"] for entry in entries if entry["backend"] == backend]
        print(f"  {backend}: {', '.join(names) if names else 'not supported'}")

In [ ]:
def run_comparison(criterion, size):
    """Run and save one criterion/size comparison."""
    bench = Benchmark(criterion=criterion)
    bench.add_all_mdps(size=size)
    bench.add_solver(backend=backends_by_criterion[criterion])

    result = bench.run(
        discount=DISCOUNT if criterion == "discounted" else None,
        precision=PRECISION,
        repeats=REPEATS,
        seed=SEED,
        timeout=TIMEOUTS[size],
        validate=VALIDATE and criterion != "average",
        verbose=True,
    )
    results[criterion, size] = result
    csv_path = result.export_csv()
    print(f"{criterion} / {size}: {csv_path}")
    print("Registered solvers:", ", ".join(result.experiment["solvers"]))

    summary = result.solver_summary()
    display(
        summary[
            [
                "solver",
                "models_solved",
                "n_models",
                "fastest_rate",
                "within_2x",
                "median_ratio",
            ]
        ].sort_values(
            ["models_solved", "within_2x", "median_ratio"],
            ascending=[False, False, True],
        )
    )
    if VALIDATE and criterion != "average":
        display(
            result.to_dataframe()
            .groupby(["solver", "validation_status"])
            .size()
            .unstack(fill_value=0)
        )
    result.plot_performance_profile()
    result.plot_heat(metric="runtime")
    return result

## Discounted reward

### tiny models

In [ ]:
result_discounted_tiny = run_comparison("discounted", "tiny")

### small models

In [ ]:
result_discounted_small = run_comparison("discounted", "small")

### medium models

In [ ]:
result_discounted_medium = run_comparison("discounted", "medium")

### large models

In [ ]:
result_discounted_large = run_comparison("discounted", "large")

## Total reward

### tiny models

In [ ]:
result_total_tiny = run_comparison("total", "tiny")

### small models

In [ ]:
result_total_small = run_comparison("total", "small")

### medium models

In [ ]:
result_total_medium = run_comparison("total", "medium")

### large models

In [ ]:
result_total_large = run_comparison("total", "large")

## Average reward

### tiny models

In [ ]:
result_average_tiny = run_comparison("average", "tiny")

### small models

In [ ]:
result_average_small = run_comparison("average", "small")

### medium models

In [ ]:
result_average_medium = run_comparison("average", "medium")

### large models

In [ ]:
result_average_large = run_comparison("average", "large")